In [10]:
import gc
import pickle
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# Two-Tower Training

## Load artifacts

In [11]:
DATA_DIR = '/kaggle/input/datasets/f0rzah/aritfacts2/'
OUTPUT_DIR = '/kaggle/working/'

train_final = pd.read_parquet(f'{DATA_DIR}train_final.parquet')
items_features = pd.read_parquet(f'{DATA_DIR}items_features.parquet')
users_features = pd.read_parquet(f'{DATA_DIR}users_features.parquet')

with open(f'{DATA_DIR}evaluation_samples.pkl', 'rb') as f:
    eval_data = pickle.load(f)

print(f'train_final: {train_final.shape}')

train_final: (9178632, 25)


In [12]:
train_sampled_users = eval_data['train_sampled_users']
val_sampled_users = eval_data['val_sampled_users']
train_ground_truth = eval_data['train_ground_truth']
val_ground_truth = eval_data['val_ground_truth']
train_candidates = eval_data['train_candidates']
val_candidates = eval_data['val_candidates']

print(f'train users: {len(train_sampled_users)}, val users: {len(val_sampled_users)}')

train users: 500, val users: 500


## Vocabularies

In [13]:
CAT_FEATURES = ['age', 'sex', 'income', 'content_type', 'release_decade']

def build_vocab(series):
    vals = series.fillna('Unknown').astype(str).unique().tolist()
    if 'Unknown' in vals:
        vals.remove('Unknown')
    return {v: i for i, v in enumerate(['Unknown'] + vals)}

vocabularies = {
    'age': build_vocab(users_features['age']),
    'sex': build_vocab(users_features['sex']),
    'income': build_vocab(users_features['income']),
    'content_type': build_vocab(items_features['content_type']),
    'release_decade': build_vocab(items_features['release_decade']),
}

vocab_sizes = {k: len(v) for k, v in vocabularies.items()}
print(vocab_sizes)

{'age': 7, 'sex': 3, 'income': 7, 'content_type': 3, 'release_decade': 7}


## Dataset preparation

In [14]:
train_positives = train_final[train_final['target'] == 1].reset_index(drop=True)
print(f'train positives: {train_positives.shape}')

train positives: (3059544, 25)


In [15]:
class TwoTowerDataset(Dataset):
    def __init__(self, df, vocabularies):
        self.user_ids = df['user_id'].values.astype(np.int64)
        self.item_ids = df['item_id'].values.astype(np.int64)

        self.u_age = df['age'].fillna('Unknown').astype(str).map(vocabularies['age']).fillna(0).values.astype(np.int64)
        self.u_sex = df['sex'].fillna('Unknown').astype(str).map(vocabularies['sex']).fillna(0).values.astype(np.int64)
        self.u_income = df['income'].fillna('Unknown').astype(str).map(vocabularies['income']).fillna(0).values.astype(np.int64)
        self.u_kids = df['kids_flg'].fillna(0).values.astype(np.float32)
        self.u_interactions = df['user_total_interactions_log'].fillna(0).values.astype(np.float32)

        self.i_type = df['content_type'].fillna('Unknown').astype(str).map(vocabularies['content_type']).fillna(0).values.astype(np.int64)
        self.i_decade = df['release_decade'].fillna('Unknown').astype(str).map(vocabularies['release_decade']).fillna(0).values.astype(np.int64)
        self.i_age_rating = df['age_rating_ord'].fillna(2).values.astype(np.float32)
        self.i_genre_score = df['genre_score'].fillna(0).values.astype(np.float32)
        self.i_country_score = df['country_score'].fillna(0).values.astype(np.float32)
        self.i_actor_score = df['actor_score'].fillna(0).values.astype(np.float32)
        self.i_director_score = df['director_score'].fillna(0).values.astype(np.float32)
        self.i_pop_log = df['item_popularity_log'].fillna(0).values.astype(np.float32)
        self.i_avg_dur_log = df['item_avg_duration_log'].fillna(0).values.astype(np.float32)
        self.i_kids_cross = df['is_kids_user_and_kids_content'].fillna(0).values.astype(np.float32)

    def __len__(self):
        return len(self.user_ids)

    def __getitem__(self, idx):
        return {
            'user_id': self.user_ids[idx],
            'item_id': self.item_ids[idx],
            'u_age': self.u_age[idx],
            'u_sex': self.u_sex[idx],
            'u_income': self.u_income[idx],
            'u_kids': self.u_kids[idx],
            'u_interactions': self.u_interactions[idx],
            'i_type': self.i_type[idx],
            'i_decade': self.i_decade[idx],
            'i_age_rating': self.i_age_rating[idx],
            'i_genre_score': self.i_genre_score[idx],
            'i_country_score': self.i_country_score[idx],
            'i_actor_score': self.i_actor_score[idx],
            'i_director_score': self.i_director_score[idx],
            'i_pop_log': self.i_pop_log[idx],
            'i_avg_dur_log': self.i_avg_dur_log[idx],
            'i_kids_cross': self.i_kids_cross[idx],
        }

In [16]:
BATCH_SIZE = 512

train_dataset = TwoTowerDataset(train_positives, vocabularies)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True,
)

print(f'batches per epoch: {len(train_loader)}')

batches per epoch: 5975


## Architecture

In [17]:
class UserTower(nn.Module):
    def __init__(self, vocab_sizes, emb_dim=16, hidden_dim=128, out_dim=64):
        super().__init__()
        self.emb_age = nn.Embedding(vocab_sizes['age'], emb_dim)
        self.emb_sex = nn.Embedding(vocab_sizes['sex'], emb_dim // 2)
        self.emb_income = nn.Embedding(vocab_sizes['income'], emb_dim)

        in_dim = emb_dim + emb_dim // 2 + emb_dim + 2
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, out_dim),
            nn.ReLU(),
        )

    def forward(self, u_age, u_sex, u_income, u_kids, u_interactions):
        x = torch.cat([
            self.emb_age(u_age),
            self.emb_sex(u_sex),
            self.emb_income(u_income),
            u_kids.unsqueeze(1),
            u_interactions.unsqueeze(1),
        ], dim=1)
        return self.mlp(x)

In [18]:
class ItemTower(nn.Module):
    def __init__(self, vocab_sizes, emb_dim=16, hidden_dim=128, out_dim=64):
        super().__init__()
        self.emb_type = nn.Embedding(vocab_sizes['content_type'], emb_dim)
        self.emb_decade = nn.Embedding(vocab_sizes['release_decade'], emb_dim)

        in_dim = emb_dim * 2 + 8
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, out_dim),
            nn.ReLU(),
        )

    def forward(self, i_type, i_decade, i_age_rating, i_genre_score, i_country_score,
                i_actor_score, i_director_score, i_pop_log, i_avg_dur_log, i_kids_cross):
        x = torch.cat([
            self.emb_type(i_type),
            self.emb_decade(i_decade),
            i_age_rating.unsqueeze(1),
            i_genre_score.unsqueeze(1),
            i_country_score.unsqueeze(1),
            i_actor_score.unsqueeze(1),
            i_director_score.unsqueeze(1),
            i_pop_log.unsqueeze(1),
            i_avg_dur_log.unsqueeze(1),
            i_kids_cross.unsqueeze(1),
        ], dim=1)
        return self.mlp(x)

In [19]:
class ProjectionHead(nn.Module):
    def __init__(self, in_dim=64, hidden_dim=32, out_dim=32):
        super().__init__()
        self.projector = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x):
        return self.projector(x)

In [20]:
class TwoTowerModel(nn.Module):
    def __init__(self, vocab_sizes, emb_dim=16, hidden_dim=128, out_dim=64, proj_dim=32):
        super().__init__()
        self.user_tower = UserTower(vocab_sizes, emb_dim, hidden_dim, out_dim)
        self.item_tower = ItemTower(vocab_sizes, emb_dim, hidden_dim, out_dim)
        self.user_projection = ProjectionHead(out_dim, out_dim // 2, proj_dim)
        self.item_projection = ProjectionHead(out_dim, out_dim // 2, proj_dim)

    def forward(self, batch, return_proj=True):
        u_emb = self.user_tower(
            batch['u_age'], batch['u_sex'], batch['u_income'],
            batch['u_kids'], batch['u_interactions'],
        )
        i_emb = self.item_tower(
            batch['i_type'], batch['i_decade'], batch['i_age_rating'],
            batch['i_genre_score'], batch['i_country_score'], batch['i_actor_score'],
            batch['i_director_score'], batch['i_pop_log'], batch['i_avg_dur_log'],
            batch['i_kids_cross'],
        )

        if return_proj:
            return u_emb, i_emb, self.user_projection(u_emb), self.item_projection(i_emb)
        return u_emb, i_emb

In [21]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_CONFIG = {'emb_dim': 16, 'hidden_dim': 128, 'out_dim': 64, 'proj_dim': 32}

model = TwoTowerModel(vocab_sizes=vocab_sizes, **MODEL_CONFIG).to(DEVICE)
print(f'params: {sum(p.numel() for p in model.parameters()):,}')

params: 33,944


In [22]:
def info_nce_loss(u_proj, i_proj, temperature=0.1):
    u_proj = F.normalize(u_proj, p=2, dim=1)
    i_proj = F.normalize(i_proj, p=2, dim=1)
    logits = torch.matmul(u_proj, i_proj.T) / temperature
    labels = torch.arange(u_proj.size(0), device=u_proj.device)
    return F.cross_entropy(logits, labels)

## Validation set

In [23]:
def build_val_pairs(val_ground_truth, users_features, items_features):
    rows = [(uid, iid) for uid, items in val_ground_truth.items() for iid in items]
    df = pd.DataFrame(rows, columns=['user_id', 'item_id'])
    df = df.merge(users_features, on='user_id', how='left')
    df = df.merge(items_features, on='item_id', how='left')
    df['is_kids_user_and_kids_content'] = (
        (df['kids_flg'] == 1) & (df['age_rating_ord'] <= 1)
    ).astype('int8')
    return df

val_positives = build_val_pairs(val_ground_truth, users_features, items_features)
val_dataset = TwoTowerDataset(val_positives, vocabularies)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'val positives: {val_positives.shape}, batches: {len(val_loader)}')

val positives: (1431, 22), batches: 3


In [24]:
def validate_model(model, loader, device):
    model.eval()
    total_loss = 0.0
    n_batches = 0
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
            _, _, u_proj, i_proj = model(batch, return_proj=True)
            total_loss += info_nce_loss(u_proj, i_proj).item()
            n_batches += 1
    model.train()
    return total_loss / max(n_batches, 1)

## Training

In [25]:
NUM_EPOCHS = 10
LR = 1e-3

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-5)

In [26]:
train_losses = []
val_losses = []
best_val_loss = float('inf')
best_state = None

for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_loss = 0.0
    start = time.time()

    for batch in train_loader:
        batch = {k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
        optimizer.zero_grad()
        _, _, u_proj, i_proj = model(batch, return_proj=True)
        loss = info_nce_loss(u_proj, i_proj)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        epoch_loss += loss.item()

    avg_train = epoch_loss / len(train_loader)
    val_loss = validate_model(model, val_loader, DEVICE)
    scheduler.step()

    train_losses.append(avg_train)
    val_losses.append(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    print(f'epoch {epoch + 1}/{NUM_EPOCHS} | train {avg_train:.4f} | val {val_loss:.4f} | {time.time() - start:.1f}s')

epoch 1/10 | train 6.1061 | val 6.0828 | 86.2s
epoch 2/10 | train 6.0921 | val 6.1084 | 85.6s
epoch 3/10 | train 6.0877 | val 6.1004 | 86.6s
epoch 4/10 | train 6.0844 | val 6.1207 | 85.4s
epoch 5/10 | train 6.0817 | val 6.0969 | 84.5s
epoch 6/10 | train 6.0794 | val 6.1013 | 84.3s
epoch 7/10 | train 6.0773 | val 6.1005 | 84.4s
epoch 8/10 | train 6.0754 | val 6.1195 | 84.5s
epoch 9/10 | train 6.0739 | val 6.1096 | 84.5s
epoch 10/10 | train 6.0732 | val 6.1190 | 85.1s


In [27]:
model.load_state_dict(best_state)
model.eval()
print(f'best val loss: {best_val_loss:.4f}')

best val loss: 6.0828


## Extract embeddings

In [28]:
def extract_user_embeddings(model, users_features, vocabularies, device, batch_size=1024):
    model.eval()
    embs, ids = [], []
    for i in tqdm(range(0, len(users_features), batch_size)):
        df = users_features.iloc[i:i + batch_size]
        n = len(df)
        batch = {
            'u_age': torch.tensor(df['age'].fillna('Unknown').astype(str).map(vocabularies['age']).fillna(0).values, dtype=torch.long, device=device),
            'u_sex': torch.tensor(df['sex'].fillna('Unknown').astype(str).map(vocabularies['sex']).fillna(0).values, dtype=torch.long, device=device),
            'u_income': torch.tensor(df['income'].fillna('Unknown').astype(str).map(vocabularies['income']).fillna(0).values, dtype=torch.long, device=device),
            'u_kids': torch.tensor(df['kids_flg'].fillna(0).values, dtype=torch.float32, device=device),
            'u_interactions': torch.tensor(df['user_total_interactions_log'].fillna(0).values, dtype=torch.float32, device=device),
            'i_type': torch.zeros(n, dtype=torch.long, device=device),
            'i_decade': torch.zeros(n, dtype=torch.long, device=device),
            'i_age_rating': torch.zeros(n, device=device),
            'i_genre_score': torch.zeros(n, device=device),
            'i_country_score': torch.zeros(n, device=device),
            'i_actor_score': torch.zeros(n, device=device),
            'i_director_score': torch.zeros(n, device=device),
            'i_pop_log': torch.zeros(n, device=device),
            'i_avg_dur_log': torch.zeros(n, device=device),
            'i_kids_cross': torch.zeros(n, device=device),
        }
        with torch.no_grad():
            u_emb, _ = model(batch, return_proj=False)
        embs.append(F.normalize(u_emb, p=2, dim=1).cpu().numpy())
        ids.extend(df['user_id'].values)
    return np.vstack(embs), np.array(ids)

In [29]:
def extract_item_embeddings(model, items_features, vocabularies, device, batch_size=1024):
    model.eval()
    embs, ids = [], []
    for i in tqdm(range(0, len(items_features), batch_size)):
        df = items_features.iloc[i:i + batch_size]
        n = len(df)
        batch = {
            'i_type': torch.tensor(df['content_type'].fillna('Unknown').astype(str).map(vocabularies['content_type']).fillna(0).values, dtype=torch.long, device=device),
            'i_decade': torch.tensor(df['release_decade'].fillna('Unknown').astype(str).map(vocabularies['release_decade']).fillna(0).values, dtype=torch.long, device=device),
            'i_age_rating': torch.tensor(df['age_rating_ord'].fillna(2).values, dtype=torch.float32, device=device),
            'i_genre_score': torch.tensor(df['genre_score'].fillna(0).values, dtype=torch.float32, device=device),
            'i_country_score': torch.tensor(df['country_score'].fillna(0).values, dtype=torch.float32, device=device),
            'i_actor_score': torch.tensor(df['actor_score'].fillna(0).values, dtype=torch.float32, device=device),
            'i_director_score': torch.tensor(df['director_score'].fillna(0).values, dtype=torch.float32, device=device),
            'i_pop_log': torch.tensor(df['item_popularity_log'].fillna(0).values, dtype=torch.float32, device=device),
            'i_avg_dur_log': torch.tensor(df['item_avg_duration_log'].fillna(0).values, dtype=torch.float32, device=device),
            'i_kids_cross': torch.zeros(n, device=device),
            'u_age': torch.zeros(n, dtype=torch.long, device=device),
            'u_sex': torch.zeros(n, dtype=torch.long, device=device),
            'u_income': torch.zeros(n, dtype=torch.long, device=device),
            'u_kids': torch.zeros(n, device=device),
            'u_interactions': torch.zeros(n, device=device),
        }
        with torch.no_grad():
            _, i_emb = model(batch, return_proj=False)
        embs.append(F.normalize(i_emb, p=2, dim=1).cpu().numpy())
        ids.extend(df['item_id'].values)
    return np.vstack(embs), np.array(ids)

In [30]:
user_embs, user_ids = extract_user_embeddings(model, users_features, vocabularies, DEVICE)
item_embs, item_ids = extract_item_embeddings(model, items_features, vocabularies, DEVICE)

user_emb_dict = {int(uid): emb for uid, emb in zip(user_ids, user_embs)}
item_emb_dict = {int(iid): emb for iid, emb in zip(item_ids, item_embs)}
print(f'user embs: {user_embs.shape}, item embs: {item_embs.shape}')

100%|██████████| 16/16 [00:00<00:00, 255.66it/s]


user embs: (840197, 64), item embs: (15963, 64)


## Metrics

In [31]:
def compute_metrics(top_k_items, true_items, k=10):
    true_set = set(true_items)
    hits = 0
    sum_precision = 0.0
    sum_ndcg = 0.0
    for rank, item in enumerate(top_k_items[:k], start=1):
        if item in true_set:
            hits += 1
            sum_precision += hits / rank
            sum_ndcg += 1.0 / np.log2(rank + 1)
    num_relevant = min(k, len(true_set))
    if num_relevant == 0:
        return 0.0, 0.0, 0.0
    map_ = sum_precision / num_relevant
    ideal_dcg = sum(1.0 / np.log2(i + 1) for i in range(1, num_relevant + 1))
    ndcg = sum_ndcg / ideal_dcg if ideal_dcg > 0 else 0.0
    hr = 1.0 if hits > 0 else 0.0
    return hr, ndcg, map_

In [32]:
def evaluate(sampled_users, candidates, ground_truth, user_emb_dict, item_emb_dict, k=10):
    hr_list, ndcg_list, map_list = [], [], []
    for uid in tqdm(sampled_users, desc='scoring'):
        uid = int(uid)
        if uid not in user_emb_dict:
            hr_list.append(0.0); ndcg_list.append(0.0); map_list.append(0.0)
            continue
        cands = [iid for iid in candidates.get(uid, []) if iid in item_emb_dict]
        if not cands:
            hr_list.append(0.0); ndcg_list.append(0.0); map_list.append(0.0)
            continue
        cand_embs = np.array([item_emb_dict[iid] for iid in cands])
        scores = cand_embs @ user_emb_dict[uid]
        top_k = [cands[i] for i in np.argsort(scores)[::-1][:k]]
        hr, ndcg, map_ = compute_metrics(top_k, ground_truth.get(uid, []), k=k)
        hr_list.append(hr); ndcg_list.append(ndcg); map_list.append(map_)
    return float(np.mean(hr_list)), float(np.mean(ndcg_list)), float(np.mean(map_list))

In [33]:
hr_train, ndcg_train, map_train = evaluate(
    train_sampled_users, train_candidates, train_ground_truth, user_emb_dict, item_emb_dict,
)

hr_val, ndcg_val, map_val = evaluate(
    val_sampled_users, val_candidates, val_ground_truth, user_emb_dict, item_emb_dict,
)

scoring: 100%|██████████| 500/500 [00:00<00:00, 3390.38it/s]


In [34]:
results = pd.DataFrame({
    'Metric': ['HR@10', 'NDCG@10', 'MAP@10'],
    'Train': [hr_train, ndcg_train, map_train],
    'Val': [hr_val, ndcg_val, map_val],
})
results['Gap'] = results['Train'] - results['Val']
print(results.to_string(index=False))

 Metric    Train      Val       Gap
  HR@10 0.122000 0.130000 -0.008000
NDCG@10 0.060817 0.049741  0.011076
 MAP@10 0.043813 0.032761  0.011053


## Save model

In [35]:
MODEL_PATH = f'{OUTPUT_DIR}two_tower_model.pt'

torch.save({
    'state_dict': best_state,
    'vocabularies': vocabularies,
    'vocab_sizes': vocab_sizes,
    'config': MODEL_CONFIG,
}, MODEL_PATH)

print(f'Model saved to {MODEL_PATH}')

Model saved to /kaggle/working/two_tower_model.pt
